In [ ]:
SESSION_UUID = "REPLACE_WITH_SESSION_UUID"

import pathlib
WORKSPACES = pathlib.Path("/home/ubuntu/lab/workspaces")
SESSION_DIR = WORKSPACES / SESSION_UUID / f"session_{SESSION_UUID}"
assert SESSION_DIR.exists(), f"Workspace not found: {SESSION_DIR}"

In [ ]:
import json
import pandas as pd

THERMAL_SEVERITY = {
    # Android
    "THERMAL_STATUS_NONE": 0, "THERMAL_STATUS_LIGHT": 1,
    "THERMAL_STATUS_MODERATE": 2, "THERMAL_STATUS_SEVERE": 3,
    "THERMAL_STATUS_CRITICAL": 4, "THERMAL_STATUS_EMERGENCY": 5,
    "THERMAL_STATUS_SHUTDOWN": 6,
    # iOS
    "nominal": 0, "fair": 1, "serious": 2, "critical": 3,
}

records = []
with open(SESSION_DIR / "frames.ndjson") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        r = json.loads(line)
        records.append({
            "frame_number"   : r["frame_number"],
            "ts_app"         : r["ts_app"],
            "measured_fps"   : r.get("measured_fps"),
            "battery_pct"    : r.get("battery_level_pct"),
            "thermal_state"  : r.get("thermal_state"),
            "thermal_severity": THERMAL_SEVERITY.get(r.get("thermal_state"), 0),
            "camera_obscured": r.get("camera_obscured", False),
            "has_gps"        : len(r.get("gps", [])) > 0,
        })

df = pd.DataFrame(records)
t0 = df["ts_app"].min()
df["t_sec"] = (df["ts_app"] - t0) / 1e9

with open(SESSION_DIR / "session_meta.json") as f:
    meta = json.load(f)

print(f"Frames: {len(df)}, Duration: {df['t_sec'].max():.1f}s")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

health = meta.get("health_summary", {})
obscuration_events = health.get("camera_obscuration_events", [])
mic_offline_events = health.get("mic_offline_events", [])
dropout_events = health.get("sensor_dropout_events", [])

fig, axes = plt.subplots(5, 1, figsize=(16, 14), sharex=True)
fig.suptitle(f"Session Timeline \u2014 {SESSION_UUID[:8]}...", fontsize=13)

# Panel 1: Frame rate
axes[0].plot(df["t_sec"], df["measured_fps"], color="steelblue", linewidth=0.8)
axes[0].axhline(15, color="gray", linestyle="--", linewidth=0.6, label="Target 15 fps")
axes[0].set_ylabel("FPS")
axes[0].set_title("Frame Rate")
axes[0].legend(fontsize=8)

# Panel 2: Battery
axes[1].plot(df["t_sec"], df["battery_pct"], color="green", linewidth=0.8)
axes[1].set_ylabel("%")
axes[1].set_title("Battery Level")
axes[1].set_ylim(0, 100)

# Panel 3: Thermal severity
axes[2].plot(df["t_sec"], df["thermal_severity"], color="orangered", linewidth=0.8)
axes[2].set_ylabel("Severity")
axes[2].set_title("Thermal State (0=nominal \u2192 6=shutdown)")
axes[2].set_ylim(-0.2, 6.2)

# Panel 4: Camera obscuration
axes[3].fill_between(df["t_sec"], df["camera_obscured"].astype(int),
                     color="red", alpha=0.5, label="Obscured")
axes[3].set_ylabel("Obscured")
axes[3].set_title("Camera Obscuration")
axes[3].set_ylim(-0.1, 1.3)

# Panel 5: GPS fix availability
axes[4].fill_between(df["t_sec"], df["has_gps"].astype(int),
                     color="purple", alpha=0.5, label="GPS fix")
axes[4].set_ylabel("Fix")
axes[4].set_title("GPS Fix Availability")
axes[4].set_xlabel("Time (s)")
axes[4].set_ylim(-0.1, 1.3)

# Overlay mic offline as shaded regions on all panels
for event in mic_offline_events:
    t_start = (event["ts_app_start"] - t0) / 1e9
    t_end   = (event["ts_app_end"]   - t0) / 1e9
    for ax in axes:
        ax.axvspan(t_start, t_end, color="yellow", alpha=0.25)

# Overlay sensor dropouts as vertical lines on frame rate panel
for event in dropout_events:
    t_drop = (event["ts_app_dropout"] - t0) / 1e9
    axes[0].axvline(t_drop, color="red", linewidth=1.2, alpha=0.7,
                    label=f"dropout:{event['sensor_type']}")

plt.tight_layout()
plt.show()

In [ ]:
print("── Session Quality Summary ──────────────────")
print(f"  Duration              : {df['t_sec'].max():.1f} s")
print(f"  Mean FPS              : {df['measured_fps'].mean():.2f}")
print(f"  Min FPS               : {df['measured_fps'].min():.2f}")
print(f"  Frames below 10 fps   : {(df['measured_fps'] < 10).sum()}")
print(f"  Obscured frames       : {df['camera_obscured'].sum()}")
print(f"  Obscuration events    : {len(obscuration_events)}")
print(f"  Mic offline events    : {len(mic_offline_events)}")
print(f"  Sensor dropout events : {len(dropout_events)}")
print(f"  GPS frames with fix   : {df['has_gps'].sum()} / {len(df)}")
print(f"  Timestamp coherence   : {health.get('timestamp_coherence_failures', 0)} failures")
print(f"  Max thermal severity  : {df['thermal_severity'].max():.0f}")